# EnergyID Dossier & Fluvius Integration URL Script

**Objective:** Automate dossier management in EnergyID and generate appropriate Fluvius integration URLs.

**Revised Workflow:**
1.  **Initialization:** Load API key and source Excel file.
2.  **Dossier Processing (Loop per CSV row):**
    a.  **Identify/Create Dossier:**
        *   Search for an existing EnergyID dossier using `Costcenter` (matched against `tags` like "ImportCode:{Costcenter}" or "{Costcenter}").
        *   If not found, create a new dossier. `Installatie_Naam` as `displayName`, `Costcenter` and `script_created_dossier` as `tags`. Other details (`City`, `Postcode`, `recordType="ProductionUnit"`, `category="Production"`, `country="BE"`) are set from CSV or defaults.
    b.  **Check Fluvius Integration:**
        *   For the identified/created dossier, fetch its integrations: `GET /api/v1/records/{dossier_id}/integrations?expand=service`.
        *   Look for an integration where `service.slug` (or `service.displayName`) indicates "Fluvius".
    c.  **Generate Fluvius URL:**
        *   **If Fluvius integration exists:** Use its `id` (e.g., `integration.id`) to form the management URL: `https://app.energyid.eu/integrations/Fluvius/{integration_id}`.
        *   **If NO Fluvius integration exists:** Use the dossier's `recordNumber` (the "EA-..." one) to form the new integration URL: `https://app.energyid.eu/integrations/Fluvius/new?recordNumber={record_number}&customerFlow=zakelijk`.
    d.  **Store URL & Data:** Add the generated URL to a new column in the DataFrame. Optionally, store other useful info from the integration (like `settings.reference`) if an integration exists.
3.  **Output:** Save the updated DataFrame to a new Excel file.
4.  **Cleanup (Optional Cell):** Provide a way to delete dossiers created by the script (e.g., those tagged with "script_created_dossier").

In [ ]:
import polars as pl
import requests

print("Libraries imported.")

In [ ]:
# Load environment variables from .env file
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path=".env")

API_KEY = os.getenv("API_KEY")
BASE_API_URL = os.getenv("BASE_API_URL")
APP_DOMAIN = os.getenv("APP_DOMAIN")
WORKSPACE_ID = os.getenv("WORKSPACE_ID")
EXCEL_FILE_PATH = os.getenv("EXCEL_FILE_PATH")
STAGING_DOMAIN = os.getenv("STAGING_DOMAIN")

display()

In [ ]:
STAGING_DOMAIN = os.getenv("STAGING_DOMAIN")

display(STAGING_DOMAIN)

In [ ]:
headers = {
    "Authorization": f"apiKey {API_KEY}",
    "Content-Type": "application/json",
    "Accept": "application/json",
}

df = None
try:
    df = pl.read_excel(EXCEL_FILE_PATH)
    # Add new columns for the output, if they don't exist
    if "FluviusURL" not in df.columns:
        df = df.with_columns(pl.lit(None).alias("FluviusURL").cast(pl.Utf8))
    if "FluviusIntegrationID" not in df.columns:
        df = df.with_columns(pl.lit(None).alias("FluviusIntegrationID").cast(pl.Utf8))
    if "FluviusReference" not in df.columns:
        df = df.with_columns(pl.lit(None).alias("FluviusReference").cast(pl.Utf8))
    if "Dossiernummer" not in df.columns:
        df = df.with_columns(pl.lit(None).alias("Dossiernummer").cast(pl.Utf8))
    if "Type gebouw" not in df.columns:
        df = df.with_columns(pl.lit(None).alias("Type gebouw").cast(pl.Utf8))

    print(f"File '{EXCEL_FILE_PATH}' loaded. Columns: {df.columns}")
except Exception as e:
    print(f"Error loading file '{EXCEL_FILE_PATH}': {e}")

## Core Functions: Dossier & Integration Handling

In [ ]:
def find_dossier_by_costcenter(all_records, costcenter_val):
    # Match on the 'importCode' field with the costcenter to check if the dossier already exists
    for record in all_records:
        if str(record.get("importCode", "")) == str(costcenter_val):
            return record
    return None


def create_dossier_api(
    installatie_naam,
    costcenter,
    postcode,
    city,
    type_gebouw=None,
    country="BE",
    record_type="ProductionUnit",
    category="Production",
):
    # tags_to_set = [f"ImportCode:{costcenter}", str(costcenter), "script_created_dossier"]
    tags_to_set = []
    if type_gebouw:
        tags_to_set.append(str(type_gebouw))
    body = {
        "displayName": installatie_naam,
        "recordType": record_type,
        "city": city,
        "postalCode": str(postcode),
        "country": country,
        "category": category,
        "tags": tags_to_set,
        "WorkspaceId": WORKSPACE_ID,
        "importCode": str(costcenter),
    }
    try:
        response = requests.post(f"{BASE_API_URL}/records", headers=headers, json=body)
        response.raise_for_status()
        new_record = response.json()
        print(
            f"  CREATED Dossier: {new_record['id']} for {costcenter} with tags {new_record.get('tags')}"
        )
        return new_record
    except requests.exceptions.RequestException as e:
        print(
            f"  ERROR creating dossier for {costcenter}: {e.response.text if e.response else e}"
        )
        return None


def get_fluvius_integration_info(record_id):
    try:
        url = f"{BASE_API_URL}/records/{record_id}/integrations?expand=service"
        response = requests.get(url, headers=headers)
        response.raise_for_status()
        integrations = response.json()
        for igr in integrations:
            service = igr.get("service", {})
            if (
                "fluvius" in service.get("slug", "").lower()
                or "fluvius" in service.get("displayName", "").lower()
            ):
                return igr  # Return the whole integration object
        return None  # No Fluvius integration found
    except requests.exceptions.RequestException as e:
        print(
            f"  ERROR fetching integrations for {record_id}: {e.response.text if e.response else e}"
        )
        return None

## Main Processing Loop

Iterate through each row of the DataFrame:
- Find or create the dossier.
- Check for Fluvius integration and generate the appropriate URL.
- Update the DataFrame with the results.

In [ ]:
if df is not None:
    all_existing_records = []
    try:
        response = requests.get(f"{BASE_API_URL}/members/me/records", headers=headers)
        response.raise_for_status()
        all_existing_records = response.json()
        print(f"Fetched {len(all_existing_records)} initial records.")
    except requests.exceptions.RequestException as e:
        print(f"ERROR fetching initial records: {e}")

    output_rows = []  # This list will store dictionaries for the new DataFrame

    for i, row in enumerate(
        df.iter_rows(named=True)
    ):  # iter_rows(named=True) gives dicts
        costcenter = row["Costcenter"]
        installatie_naam = row["Installatie_Naam"]
        postcode = row["Postcode"]
        city = row["Plaats"]
        type_gebouw = row.get("Type gebouw") if "Type gebouw" in row else None

        print(
            f"\nProcessing {i + 1}/{len(df)}: Costcenter {costcenter} - {installatie_naam}"
        )

        current_dossier = find_dossier_by_costcenter(all_existing_records, costcenter)

        if not current_dossier:
            print(f"  Dossier for {costcenter} not found. Creating...")
            current_dossier = create_dossier_api(
                installatie_naam, costcenter, postcode, city, type_gebouw=type_gebouw
            )
            if current_dossier:
                all_existing_records.append(current_dossier)
            else:
                print(f"  Failed to create dossier for {costcenter}. Skipping.")
                # Add original row data + error indication to output_rows
                output_rows.append(
                    {
                        **row,
                        "FluviusURL": "ERROR_CREATING_DOSSIER",
                        "Dossiernummer": None,
                        "FluviusIntegrationID": None,
                        "FluviusReference": None,
                    }
                )
                continue
        else:
            print(f"  FOUND Dossier: {current_dossier['id']} for {costcenter}")

        dossier_id = current_dossier["id"]
        record_number = current_dossier.get("recordNumber")

        # Format Dossiernummer as EA-{record_number} (8 digits, zero-padded if needed)
        if record_number and isinstance(record_number, str):
            digits = "".join(filter(str.isdigit, record_number))
            dossiernummer_val = f"EA-{digits.zfill(8)}" if digits else None
        else:
            dossiernummer_val = None

        fluvius_igr_obj = get_fluvius_integration_info(dossier_id)
        fluvius_url_val = None
        fluvius_integration_id_val = None
        fluvius_reference_val = None

        if fluvius_igr_obj:
            integration_id = fluvius_igr_obj["id"]
            fluvius_url_val = f"{STAGING_DOMAIN}/integrations/Fluvius/{integration_id}"
            fluvius_integration_id_val = integration_id
            fluvius_reference_val = fluvius_igr_obj.get("settings", {}).get("reference")
            print(
                f"  EXISTING Fluvius integration found (ID: {integration_id}). URL: {fluvius_url_val}"
            )
        elif record_number:
            fluvius_url_val = f"{STAGING_DOMAIN}/integrations/Fluvius/new?recordNumber={record_number}"
            print(f"  NO Fluvius integration. NEW URL: {fluvius_url_val}")
        else:
            print(
                f"  NO Fluvius integration and no recordNumber for dossier {dossier_id}. Cannot generate new URL."
            )
            fluvius_url_val = "ERROR_MISSING_RECORD_NUMBER"

        # Append a dictionary representing the full row for the new DataFrame
        # This includes all original columns from 'row' and the new/updated ones
        output_rows.append(
            {
                **row,
                "Dossiernummer": dossiernummer_val,
                "FluviusURL": fluvius_url_val,
                "FluviusIntegrationID": fluvius_integration_id_val,
                "FluviusReference": fluvius_reference_val,
            }
        )

    if output_rows:
        df_updated = pl.DataFrame(output_rows)

        # Place Dossiernummer right after the original columns, then the other new columns
        original_cols = list(df.columns)
        new_cols = [
            col
            for col in [
                "Dossiernummer",
                "FluviusURL",
                "FluviusIntegrationID",
                "FluviusReference",
            ]
            if col not in original_cols
        ]
        # Ensure Dossiernummer is always first among new columns
        if "Dossiernummer" in new_cols:
            new_cols.remove("Dossiernummer")
            final_column_names = original_cols + ["Dossiernummer"] + new_cols
        else:
            final_column_names = original_cols + new_cols

        # Select columns in the desired order, dropping any from df_updated not in final_column_names (shouldn't happen here)
        df_final_output = df_updated.select(final_column_names)

        print("\n--- Final DataFrame with Fluvius URLs ---")
        print(df_final_output.head())

        try:
            output_excel_path = EXCEL_FILE_PATH.replace(
                ".xlsx", "_with_fluvius_urls.xlsx"
            )
            df_final_output.write_excel(output_excel_path)
            print(f"\nUpdated data saved to: {output_excel_path}")
        except Exception as e:
            print(f"Error saving output Excel: {e}")
    else:
        print("No rows processed to output.")

else:
    print("DataFrame not loaded. Cannot process.")

In [ ]:
# Test the first 3 rows for dossier creation and print diagnostics
def test_first_3_rows_dossier_creation():
    test_rows = list(df.iter_rows(named=True))[:3]
    for i, row in enumerate(test_rows):
        costcenter = row["Costcenter"]
        installatie_naam = row["Installatie_Naam"]
        postcode = row["Postcode"]
        city = row["Plaats"] if "Plaats" in row else row.get("City")
        type_gebouw = row.get("Type gebouw") if "Type gebouw" in row else None
        print(f"\nTesting row {i + 1}: Costcenter {costcenter} - {installatie_naam}")
        tags_to_set = [
            f"ImportCode:{costcenter}",
            str(costcenter),
            "script_created_dossier",
        ]
        if type_gebouw:
            tags_to_set.append(str(type_gebouw))
        body = {
            "displayName": installatie_naam,
            "recordType": "ProductionUnit",
            "city": city,
            "postalCode": str(postcode),
            "country": "BE",
            "category": "Production",
            "tags": tags_to_set,
            "WorkspaceId": WORKSPACE_ID,
        }
        print("Request JSON body:", body)
        print("Headers:", headers)
        try:
            response = requests.post(
                f"{BASE_API_URL}/records", headers=headers, json=body
            )
            print(f"Status code: {response.status_code}")
            print(f"Response text: {response.text}")
            if response.status_code == 400:
                print("400 Bad Request - check which parameter is causing the issue.")
            elif response.status_code >= 300:
                print("Non-success status code returned.")
            else:
                print("Success! Response JSON:")
                print(response.json())
        except Exception as e:
            print(f"Exception occurred: {e}")


test_first_3_rows_dossier_creation()

add dossiernumber,url naar aan-te-maken nummer


nice to have:
iets van info toevoegen aan output url over : meters info 
    kan pas na fluvius koppeling
    last meter reading
    timestamp
    "gegevens aangevuld tot"

In [ ]:
def delete_script_created_dossiers(tag_to_identify="script_created_dossier"):
    print(f"\n--- Deleting Dossiers Tagged: '{tag_to_identify}' ---")
    deleted_count = 0
    try:
        response = requests.get(f"{BASE_API_URL}/members/me/records", headers=headers)
        response.raise_for_status()
        all_records = response.json()
        for record in all_records:
            if tag_to_identify in record.get("tags", []):
                print(f"  Deleting {record['id']} ({record['displayName']})...")
                del_resp = requests.delete(
                    f"{BASE_API_URL}/records/{record['id']}", headers=headers
                )
                if (
                    del_resp.status_code == 200 or del_resp.status_code == 204
                ):  # 200 or 204 for successful delete
                    print(f"    DELETED {record['id']}")
                    deleted_count += 1
                else:
                    print(
                        f"    ERROR deleting {record['id']}: {del_resp.status_code} - {del_resp.text}"
                    )
        print(f"Deleted {deleted_count} dossiers.")
    except requests.exceptions.RequestException as e:
        print(f"  ERROR fetching records for cleanup: {e}")

In [ ]:
# To run cleanup (uncomment below):
# delete_script_created_dossiers()
# delete_script_created_dossiers(tag_to_identify="PV00014") # If Costcenter was used as a direct tag